In [ ]:
import pandas as pd
from datetime import *
import os 
import shutil
import pandas as pd
import numpy as np
from scipy import stats
import functools
from time import time
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import plotly.graph_objects as go
import re
from scipy.signal import find_peaks as fp

# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES, LABEL_DF

SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}
COL_Prefixes ={"HeartRate":"HR_","Metadata":"MD_","WatchAccelerometer":"ACC_", "WatchGravity":"G_","WatchGyroscope":"GYRO_","WatchOrientation":"OR_","WatchTotalAcceleration":"TACC_","WatchMagnetometer":"MAG_"}

RAWDATAFILES={}

MERGE_FILES=1

pd.set_option('display.max_rows', 500)

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
PREPROC_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"250MS_labeled.pkl"))

PREPROC_DF.iloc[0:19]



In [ ]:
# Compressed Base Features + Ground Truth
# Mean, Std, fft
# Window: 5S = 5*4*250ms = 20 slices
# Class assignment Threshold:2.5s (>=50%, >=10 slices)
feature_cols = PREPROC_DF.columns[:-5]
target_cols = PREPROC_DF.columns[-5:]

fs = 4  # Sampling frequency (Hz)
duration = 5  # seconds

def first_non_nan(empty_list):
    for element in empty_list:
        if not np.isnan(element):
            return element
    return None



base_features=[]
for col in feature_cols:
    base_features.append("mean_"+col)
    base_features.append("std_"+col)
    base_features.append("freq_"+col)
    
Feature_DF=pd.DataFrame(columns=base_features + ["target","datetime"],index=range(0,int(abs((PREPROC_DF.shape[0]-20)/20))))

n_row =0

for i in range(0,PREPROC_DF.shape[0]-40,20):

    for feature in feature_cols:
        # print(feature)
        # print(n_row)
        Feature_DF.iloc[n_row]["mean_"+feature]= PREPROC_DF.iloc[i:i+19][feature].mean()
        Feature_DF.iloc[n_row]["std_"+feature]= PREPROC_DF.iloc[i:i+19][feature].std()
        #freqency
        signal = PREPROC_DF.iloc[i:i+19][feature]
        fft_result = np.fft.fft(signal)
        # measure repetitions with amplitude higher than 20
        Feature_DF.iloc[n_row]["freq_"+feature]= len([ x for x in np.abs(fft_result) if x>=20]) 
        if len(list(PREPROC_DF.iloc[i:i+19]["target"].notna()))>=10:
            Feature_DF.iloc[n_row]["target"] = list(PREPROC_DF.iloc[i:i+19]["target"].notna())[0]
            Feature_DF.iloc[n_row]["target_name"] = list(PREPROC_DF.iloc[i:i+19]["target"].notna())[0]
        Feature_DF.iloc[n_row]["datetime"] = PREPROC_DF.index[i]
    n_row +=1


In [ ]:
#Validation
print(list(PREPROC_DF.iloc[0:19]["target"].notna()))
print(first_non_nan([2,np.nan,3,np.nan]))
#print(PREPROC_DF.loc["2025-12-22 11:38:58.250":"2025-12-22 11:39:05.250"])
#Feature_DF["datetime"] = pd.to_datetime(Feature_DF["datetime"])
#Feature_DF[(Feature_DF["datetime"]  >= pd.to_datetime("2025-12-19 15:19:47.750")) & (Feature_DF["datetime"]  <= pd.to_datetime("2025-12-19 15:20:06.750"))]

#Feature_DF.dropna()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generating a sample musical note signal
fs = 4  # Sampling frequency (Hz)
duration = 5  # seconds
frequency = 8  # A4 note frequency (Hz)
t = np.linspace(0, duration, int(fs * duration), endpoint=False)
#positives
signal = PREPROC_DF.loc["2025-12-27 16:52:00":"2025-12-27 16:52:5"]["G_x"]
#signal = PREPROC_DF.loc["2025-12-27 16:26:27":"2025-12-27 16:26:32"]["G_y"]

#negatives
#signal = PREPROC_DF.loc["2025-12-27 16:51:21":"2025-12-27 16:51:26"]["G_x"]
#signal = PREPROC_DF.loc["2025-12-27 16:25:27":"2025-12-27 16:25:32"]["G_y"]


plt.plot(signal)
plt.show()

# Applying FFT
fft_result = np.fft.fft(signal)
freq = np.fft.fftfreq(24, d=1/fs)
plt.plot(fft_result)
plt.show()

print( len([ x for x in np.abs(fft_result) if x>=20]) )


# Plotting the spectrum
plt.plot(freq, np.abs(fft_result))
plt.title('FFT of a Musical Note')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.show()

In [ ]:
PREPROC_DF.loc["2025-12-27 16:51:56":"2025-12-27 16:52:01"]